In [20]:
import kfp
from google.cloud import aiplatform
from google_cloud_pipeline_components._implementation.model import GetVertexModelOp

In [61]:
PROJECT_ID = "project-c8d59959-80e8-496f-96a"
GCP_REGION = "us-central1"
DISPLAY_NAME = "custom-credit-card-15Jul26-1"
MACHINE_TYPE = "n1-standard-4"
BUCKET_URI = "gs://custom-trin-bucket"

In [108]:
@kfp.dsl.component(
    packages_to_install=["google-cloud-aiplatform"],
    base_image="python:3.12",
)
def eval_automl_model(
    project: str,
    region: str,
) -> float:
    from google.cloud import aiplatform
    aiplatform.init(project=project, location=region)

    auto_ml_model = aiplatform.Model('projects/90514214177/locations/us-central1/models/1809286515679821824')

    automl_evaluations = auto_ml_model.list_model_evaluations()
    
    for evaluation in automl_evaluations:
        evaluation = evaluation.to_dict()
        metrics = evaluation["metrics"]
        automl_acc = metrics["auPrc"]
            
    return automl_acc

@kfp.dsl.component(
    packages_to_install=["google-cloud-aiplatform", "tensorflow", "pandas", "fsspec", "gcsfs"],
    base_image="python:3.12",
)
def eval_custom_model(
    project: str,
    region: str,
) -> float:
    from google.cloud import aiplatform
    import tensorflow as tf
    import pandas as pd

    aiplatform.init(project=project, location=region)

    custom_model = tf.keras.models.load_model('gs://custom-trin-bucket/custom_model_test1.keras')

    df = pd.read_csv("gs://bucket-for-kpf-project/test_UCI_Credit_Card.csv")
    test_labels = df.pop('default.payment.next.month')

    loss, custom_acc = custom_model.evaluate(df, test_labels, verbose=0)
            
    return custom_acc

@kfp.dsl.component
def print_comp(text: str):
    print(text)

    

/var/tmp/ipykernel_3659/4178154143.py:46: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.11' to 'python:3.12' on Oct 1, 2027. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.12.
  @kfp.dsl.component


In [109]:
@kfp.dsl.pipeline(
    name=DISPLAY_NAME,
    pipeline_root=BUCKET_URI,
    )
def pipeline(
    project: str = PROJECT_ID,
    region: str = GCP_REGION,
):

    automl_model_acc = eval_automl_model(project=project, region=region)

    custom_model_acc = eval_custom_model(project=project, region=region)
    
    with kfp.dsl.If(automl_model_acc.output > custom_model_acc.output):
        print_comp(text = f"AutmoML is better, acc = {automl_model_acc.output}")
    with kfp.dsl.Else():
        print_comp(text = f"Custom is better, acc = {custom_model_acc.output}")

In [110]:
kfp.compiler.Compiler().compile(
    pipeline_func=pipeline,
    package_path='eval_pipeline_15jul_v2.yaml'
)
!gsutil cp -r eval_pipeline_15jul_v2.yaml gs://custom-trin-bucket/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://eval_pipeline_15jul_v2.yaml [Content-Type=application/yaml]...
/ [1 files][ 11.7 KiB/ 11.7 KiB]                                                
Operation completed over 1 objects/11.7 KiB.                                     


In [111]:
aiplatform.init(
    project=PROJECT_ID,
    location=GCP_REGION,
)

# Prepare the pipeline job
job = aiplatform.PipelineJob(
    display_name=DISPLAY_NAME,
    template_path="eval_pipeline_15jul_v2.yaml",
    pipeline_root=BUCKET_URI,
    #parameter_values={
    #    "PROJECT_ID": PROJECT_ID,
    #},
    enable_caching=True,
)

job.submit()

Creating PipelineJob
PipelineJob created. Resource name: projects/90514214177/locations/us-central1/pipelineJobs/custom-credit-card-15jul26-1-20260716132233
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/90514214177/locations/us-central1/pipelineJobs/custom-credit-card-15jul26-1-20260716132233')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/custom-credit-card-15jul26-1-20260716132233?project=90514214177
